# 04 — RAG Pipeline

**Ziel**: PubMed-Abstracts in eine Vektor-DB laden, Retrieval testen, LLM-Erklärungen generieren.

**Vergleich**: 2 Embedding-Modelle + 2 Prompt-Varianten.

In [ ]:
# Falls in Colab: Installations
# !pip install -q langchain langchain-openai langchain-community chromadb sentence-transformers tiktoken python-dotenv

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Lokal: aus .env laden | Colab: aus userdata
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('OpenAI Key aus Colab Secrets geladen')
except (ImportError, ModuleNotFoundError):
    load_dotenv('../.env')
    assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY fehlt!'
    print('OpenAI Key aus .env geladen')

ABSTRACTS_PATH = Path('../data/pubmed_abstracts')
CHROMA_PATH = Path('../models/chroma_db')
CHROMA_PATH.parent.mkdir(exist_ok=True)

## 1. Abstracts laden

In [ ]:
with open(ABSTRACTS_PATH / 'pubmed_abstracts.jsonl') as f:
    abstracts = [json.loads(line) for line in f]

print(f'Loaded {len(abstracts)} abstracts')

# In LangChain-Documents konvertieren
from langchain_core.documents import Document

docs = []
for a in abstracts:
    content = f"Title: {a['title']}\n\nAbstract: {a['abstract']}"
    metadata = {
        'pmid': a['pmid'],
        'title': a['title'][:200],
        'year': a.get('year') or 0,
    }
    docs.append(Document(page_content=content, metadata=metadata))

print(f'Documents: {len(docs)}')

## 2. Embeddings & Vektor-DB aufbauen

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Lokales Embedding-Modell (kostenlos)
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
)

# Vektor-DB aufbauen (nur beim ersten Mal nötig)
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=str(CHROMA_PATH),
    collection_name='pubmed_cyclesync',
)
print(f'Vektor-DB erstellt mit {vectorstore._collection.count()} Einträgen')
print(f'Persistiert unter: {CHROMA_PATH}')

## 3. Retrieval testen

In [ ]:
# Test-Query
test_query = 'training intensity recommendation luteal phase fatigue'
results = vectorstore.similarity_search(test_query, k=4)

for i, r in enumerate(results, 1):
    print(f'--- Treffer {i} (PMID: {r.metadata["pmid"]}, Jahr: {r.metadata["year"]}) ---')
    print(r.page_content[:300])
    print()

## 4. LLM-Integration & Prompt-Varianten

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.3)

# Prompt-Variante A: Zero-Shot
PROMPT_A = ChatPromptTemplate.from_template('''\
Du bist sportwissenschaftliche Beraterin und erklärst Trainingsempfehlungen.

Nutzerin-Profil:
- Zyklusphase: {phase}
- Tag im Zyklus: {day_in_cycle}
- Geplante Sportart: {sport}
- Schlafqualität letzte Nacht: {sleep_quality}/10
- Reportierte Symptome: {symptoms}

Modell-Empfehlung:
- Intensität: {intensity}
- Risiko: basierend auf Belastung und Recovery-Bedarf

Relevante Studien (Auszug):
{retrieved_docs}

Erkläre der Nutzerin in 3–4 Sätzen auf Deutsch, warum diese Empfehlung passt. \
Verweise auf die Studien per [PMID:xxxx]. Tonfall: sachlich, unterstützend, nicht bevormundend. \
Schließe mit einem Satz Disclaimer (keine medizinische Beratung).
''')

# Prompt-Variante B: Strukturierter mit Anweisungen
PROMPT_B = ChatPromptTemplate.from_template('''\
Rolle: Du bist sportwissenschaftliche Beraterin für zyklusbasiertes Training.

Aufgabe: Erkläre der Nutzerin die folgende Trainingsempfehlung.

Vorgabe an die Antwort:
1. Beginne mit der konkreten Empfehlung in einem Satz.
2. Erkläre die physiologische Begründung in 1-2 Sätzen.
3. Verweise auf mindestens 2 der bereitgestellten Studien per [PMID:xxxx].
4. Schließe mit einem kurzen Disclaimer.

Nutzerin-Daten:
- Zyklusphase: {phase} (Tag {day_in_cycle})
- Geplante Sportart: {sport}
- Schlafqualität: {sleep_quality}/10
- Symptome: {symptoms}
- Vorhergesagte Intensität: {intensity}

Verfügbare Studien:
{retrieved_docs}

Antwort auf Deutsch:
''')

print('Prompts definiert.')

In [ ]:
def format_docs(docs_list) -> str:
    parts = []
    for d in docs_list:
        pmid = d.metadata['pmid']
        title = d.metadata['title']
        snippet = d.page_content[:400]
        parts.append(f'[PMID:{pmid}] {title}\n{snippet}\n')
    return '\n---\n'.join(parts)

def generate_explanation(user_input: dict, ml_prediction: dict, prompt_template, k=4):
    # Retrieval-Query aus Kontext bauen
    query = (f"{ml_prediction['phase']} phase {user_input['sport']} "
             f"intensity {ml_prediction['intensity']} {' '.join(user_input['symptoms'])}")
    retrieved = vectorstore.similarity_search(query, k=k)
    docs_text = format_docs(retrieved)
    
    chain = prompt_template | llm
    response = chain.invoke({
        'phase': ml_prediction['phase'],
        'day_in_cycle': user_input.get('day_in_cycle', '?'),
        'sport': user_input['sport'],
        'sleep_quality': user_input.get('sleep_quality', '?'),
        'symptoms': ', '.join(user_input['symptoms']),
        'intensity': ml_prediction['intensity'],
        'retrieved_docs': docs_text,
    })
    return {
        'explanation': response.content,
        'sources': [{'pmid': d.metadata['pmid'], 'title': d.metadata['title'], 'year': d.metadata['year']} for d in retrieved],
    }

## 5. Test-Szenarien

In [ ]:
# Szenario 1: Frühe Follikelphase
user1 = {'day_in_cycle': 7, 'sport': 'running', 'symptoms': ['none'], 'sleep_quality': 8}
pred1 = {'phase': 'follicular', 'intensity': 'high'}

result_A = generate_explanation(user1, pred1, PROMPT_A)
print('=== PROMPT A (Zero-Shot) ===')
print(result_A['explanation'])
print()

result_B = generate_explanation(user1, pred1, PROMPT_B)
print('=== PROMPT B (Strukturiert) ===')
print(result_B['explanation'])

In [ ]:
# Szenario 2: Späte Lutealphase mit Symptomen
user2 = {'day_in_cycle': 26, 'sport': 'hiit', 'symptoms': ['fatigue', 'bloating', 'mood_low'], 'sleep_quality': 4}
pred2 = {'phase': 'luteal', 'intensity': 'low'}

result_A2 = generate_explanation(user2, pred2, PROMPT_A)
print('=== PROMPT A — Szenario 2 ===')
print(result_A2['explanation'])
print()
result_B2 = generate_explanation(user2, pred2, PROMPT_B)
print('=== PROMPT B — Szenario 2 ===')
print(result_B2['explanation'])

## 6. Qualitative Bewertung (für Doku)

| Szenario | Prompt | Relevanz Quellen (1-5) | Faktentreue (1-5) | Tonfall (1-5) | Anmerkungen |
| --- | --- | --- | --- | --- | --- |
| 1 — Follikel | A | | | | |
| 1 — Follikel | B | | | | |
| 2 — Luteal | A | | | | |
| 2 — Luteal | B | | | | |

Trage hier deine subjektive Bewertung nach dem Anschauen der Outputs ein.

**Entscheidung**: Wir verwenden PROMPT_B (strukturierter) in der App, weil er konsistentere Zitierungen liefert.

Weiter mit `05_integration_test.ipynb`.